# 03 — Risk-Neutral NIG Calibration

This notebook applies the calibration engine in `src/calibration.py` to the cleaned SPX European call sample. It presents the objective, parameter constraints, estimated parameters and fit diagnostics. The numerical implementation remains in the Python module, not in notebook cells.


## 1. Objective function and identifiable parameters

Following the assignment, we minimise the unweighted mean-square price error:

`MSE(alpha,beta,delta) = (1/N) * sum_i (C_model_i - C_market_i)^2`.

The mean-correcting risk-neutral construction removes `mu` from the option-price characteristic function. We therefore estimate `(alpha, beta, delta)` only. The admissible region is `alpha > 0`, `delta > 0`, `abs(beta) < alpha`, and `abs(beta + 1) < alpha`.


## 2. Load the cleaned calibration sample

Load the compact, long-format file committed in `data/cleaned/`. Each observation is one call quote. The calibration engine checks all required columns and rejects missing or non-finite inputs.


In [2]:
from pathlib import Path
import sys

cwd = Path.cwd()
PROJECT_ROOT = next(
    (folder for folder in (cwd, *cwd.parents)
     if (folder / "src" / "calibration.py").is_file()),
    None,
)

project_path = str(PROJECT_ROOT)
if project_path not in sys.path:
    sys.path.insert(0, project_path)

import numpy as np
import pandas as pd
from src.calibration import NIGCalibrator

DATA_PATH = PROJECT_ROOT / 'data' / 'cleaned' / 'calibration_candidates.csv.gz'
if not DATA_PATH.exists():
    raise FileNotFoundError(f'Calibration candidates not found: {DATA_PATH}')
options = pd.read_csv(DATA_PATH)
print(f'Calibration observations: {len(options):,}')
print(f'Distinct expiries: {options["Expiry"].nunique()}')
display(options.groupby('Expiry').size().rename('Number of calls').to_frame())


Calibration observations: 477
Distinct expiries: 5


,Number of calls
Expiry,
2026-11-20,149
2026-12-18,145
2027-01-15,112
2027-04-16,41
2027-09-17,30


## 3. Configure and run the calibration

The optimizer works in transformed coordinates so every trial parameter set respects the risk-neutral exponential-moment constraints. The vectorised Gauss-Legendre pricing routine evaluates all options together at each objective-function call. The starting values are numerical initial guesses, not assumed final estimates.


In [3]:
calibrator = NIGCalibrator(
    integration_limit=1000.0,
    quadrature_points=512,
    max_nfev=500,
)
calibration_result = calibrator.calibrate(
    options,
    initial_parameters=(8.0, -2.0, 0.25),
)
print(calibration_result)

import pickle

RESULT_PATH = PROJECT_ROOT / "results" / "calibration_result.pkl"
RESULT_PATH.parent.mkdir(parents=True, exist_ok=True)

with RESULT_PATH.open("wb") as file:
    pickle.dump(calibration_result, file)

print(f"Saved: {RESULT_PATH}")

CalibrationResult(parameters=NIGParameters(alpha=13.527722472498198, beta=-9.525035530742965, delta=0.20862148989303406, mu=0.0), mean_squared_error=98.34431035802956, root_mean_squared_error=9.916869987956359, success=True, message='`ftol` termination condition is satisfied.', function_evaluations=12, number_of_quotes=477)
Saved: c:\Users\l67914\Downloads\NIG-SPX-Option-Pricing-setup-project-structure\NIG-SPX-Option-Pricing-setup-project-structure\results\calibration_result.pkl


## 4. Report estimated parameters and fit statistics

The table reports the estimated risk-neutral NIG parameters and the in-sample MSE/RMSE. A successful optimizer termination is necessary but not sufficient: the next cells inspect residuals and pricing errors by maturity before we interpret the fit.


In [5]:
estimated = calibration_result.parameters
parameter_summary = pd.Series({
    'alpha': estimated.alpha,
    'beta': estimated.beta,
    'delta': estimated.delta,
    'mu (not identified under this convention)': estimated.mu,
    'MSE': calibration_result.mean_squared_error,
    'RMSE': calibration_result.root_mean_squared_error,
    'Number of quotes': calibration_result.number_of_quotes,
    'Function evaluations': calibration_result.function_evaluations,
    'Optimizer success': calibration_result.success,
}, name='Estimate')
display(parameter_summary.to_frame())
print(calibration_result.message)


,Estimate
alpha,13.527722
beta,-9.525036
delta,0.208621
mu (not identified under this convention),0.0
MSE,98.34431
RMSE,9.91687
Number of quotes,477
Function evaluations,12
Optimizer success,True


`ftol` termination condition is satisfied.


## 5. Inspect model prices and residuals

Reprice the full sample using the estimated parameters. Residuals are defined as model price minus market mid-price. Inspect both the aggregate error and the maturity-level distribution; large residuals may reveal model misspecification, noisy quotes or numerical integration issues.


In [6]:
model_prices = calibrator.model_prices(
    options,
    estimated.alpha,
    estimated.beta,
    estimated.delta,
)
fit = options.copy()
fit['Model_Price'] = model_prices
fit['Pricing_Error'] = fit['Model_Price'] - fit['Market_Price']
fit['Absolute_Error'] = fit['Pricing_Error'].abs()
maturity_diagnostics = fit.groupby('Expiry').agg(
    Number_of_Quotes=('Pricing_Error', 'size'),
    Mean_Error=('Pricing_Error', 'mean'),
    MAE=('Absolute_Error', 'mean'),
    RMSE=('Pricing_Error', lambda values: float(np.sqrt(np.mean(values**2)))),
)
display(maturity_diagnostics)
display(fit[['Expiry','Strike','Market_Price','Model_Price','Pricing_Error']].head(10))


,Number_of_Quotes,Mean_Error,MAE,RMSE
Expiry,,,,
2026-11-20,149,-7.213771,8.215176,11.217335
2026-12-18,145,-3.145316,3.970106,4.650057
2027-01-15,112,2.324110,2.455523,2.500528
2027-04-16,41,11.293927,14.121767,15.359713
2027-09-17,30,-3.189530,19.456554,22.100819


,Expiry,Strike,Market_Price,Model_Price,Pricing_Error
0,2026-11-20,7000,666.050049,669.065529,3.015480
1,2026-11-20,7090,581.149902,584.091096,2.941194
2,2026-11-20,7100,571.500000,574.705986,3.205986
3,2026-11-20,7110,562.750000,565.333423,2.583423
4,2026-11-20,7150,525.399902,527.976595,2.576693
5,2026-11-20,7200,479.700073,481.614472,1.914399
6,2026-11-20,7220,461.549927,463.186746,1.636819
7,2026-11-20,7230,452.549927,454.000355,1.450429
8,2026-11-20,7250,434.299927,435.685875,1.385948
9,2026-11-20,7285,403.599853,403.838211,0.238357


## 6. Calibration checkpoint

Do not treat the parameter estimates as final until we have checked sensitivity to initial values and integration settings, compared fitted NIG prices with Black–Scholes prices calibrated to the same market sample, and calculated the assignment's ARPE measure. Those robustness and benchmark steps follow after this first calibration pass.
